# Poglavje 7: Izdelava klepetalnih aplikacij
## Hiter začetek z Microsoft Foundry Models API

Ta zvezek je prilagojen iz [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ki vsebuje zvezke za dostop do storitev [Azure OpenAI](notebook-azure-openai.ipynb).

> **Opomba:** GitHub Models bo prenehal delovati konec julija 2026. Ta zvezek sedaj cilja na [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), ki ponuja enak brezplačen katalog modelov za preizkus in izkušnjo Azure AI Inference SDK.


# Pregled  
"Veliki jezikovni modeli so funkcije, ki preslikajo besedilo v besedilo. Glede na vhodni niz besedila, velik jezikovni model poskuša napovedati, kakšno besedilo bo sledilo"(1). Ta zvezek "hitri začetek" bo uporabnike uvedel v osnovne koncepte LLM, osnovne zahteve paketov za začetek z AML, nežno uvedbo v oblikovanje pozivov ter več kratkih primerov različnih primerov uporabe. 


## Kazalo vsebine  

[Pregled](#overview)  
[Kako uporabljati storitev OpenAI](#how-to-use-openai-service)  
[1. Ustvarjanje vaše storitve OpenAI](#1.-creating-your-openai-service)  
[2. Namestitev](#2.-installation)    
[3. Pooblastila](#3.-credentials)  

[Uporabe](#use-cases)    
[1. Povzemanje besedila](#1.-summarize-text)  
[2. Klasifikacija besedila](#2.-classify-text)  
[3. Ustvarjanje novih imen izdelkov](#3.-generate-new-product-names)  
[4. Fino prilagajanje klasifikatorja](#4.fine-tune-a-classifier)  

[Reference](#references)


### Ustvarite svoj prvi poziv  
Ta kratek vajniški primer bo zagotovil osnovni uvod v pošiljanje pozivov modelu v Microsoft Foundry Models za preprosto nalogo "povzemanje".  


**Koraki**:  
1. Namestite knjižnico `azure-ai-inference` v vašem Python okolju, če je še niste.  
2. Naložite standardne pomožne knjižnice in nastavite svoje poverilnice za Microsoft Foundry Models.  
3. Izberite model za svojo nalogo  
4. Ustvarite preprost poziv za model  
5. Pošljite svojo zahtevo preko API-ja modela!  


### 1. Namestite `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Uvozi pomožne knjižnice in ustvari poverilnice


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Iskanje pravega modela  
Modeli, kot sta GPT-4o in GPT-4o mini, lahko razumejo in ustvarjajo naravni jezik ter so na voljo v katalogu Microsoft Foundry Models skupaj z modeli iz Meta, Mistral, Cohere in Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Oblikovanje poziva  

"Čar velikih jezikovnih modelov je v tem, da se z učenjem minimizacije te napake napovedi na ogromnih količinah besedila modeli na koncu naučijo pojmov, uporabnih za te napovedi. Na primer, naučijo se pojmov, kot so"(1):

* kako črkovati
* kako deluje slovnica
* kako parafrazirati
* kako odgovarjati na vprašanja
* kako voditi pogovor
* kako pisati v mnogih jezikih
* kako programirati
* itd.

#### Kako nadzorovati velik jezikovni model  
"Od vseh vhodov v velik jezikovni model je daleč najvplivnejši besedilni poziv(1).

Velike jezikovne modele lahko spodbudimo k ustvarjanju izhodov na nekaj načinov:

Navodilo: Povejte modelu, kaj želite
Dokončanje: Napeljite model, da dokonča začetek tega, kar želite
Demonstracija: Pokažite modelu, kaj želite, z:
Nekaj primeri v pozivu
Stotini ali tisoči primerov v učni zbirki za fino prilagajanje"



#### Obstajajo tri osnovna pravila za ustvarjanje pozivov:

**Pokaži in povej**. Jasno pokažite, kaj želite, bodisi z navodili, primeri ali kombinacijo obeh. Če želite, da model razvrsti seznam predmetov po abecedi ali razvrsti odstavek glede na sentiment, pokažite, da je to, kar želite.

**Posredujte kakovostne podatke**. Če poskušate zgraditi klasifikator ali modelu slediti vzorcu, se prepričajte, da je dovolj primerov. Pazljivo preglejte vaše primere — model je običajno dovolj pameten, da vidi osnovne pravopisne napake in vam odgovori, vendar lahko tudi predpostavi, da je to namenjeno, kar lahko vpliva na odgovor.

**Preverite nastavitve.** Nastavitve temperature in top_p nadzorujejo, kako determinističen je model pri generiranju odgovora. Če zahtevate odgovor, kjer je samo en pravi odgovor, želite ti nastavitvi nastaviti nižje. Če želite bolj raznolike odgovore, ju nastavite višje. Najpogostejša napaka pri teh nastavitvah je, da ljudje mislijo, da so to nastavitve "pametnosti" ali "kreativnosti".


Vir: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Oddaj!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ponovite isti klic, kako se rezultati primerjajo?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Povzetek besedila  
#### Izziv  
Povzemite besedilo tako, da na konec besedilnega odstavka dodate 'tl;dr:'. Opazite, kako model razume, kako izvesti več nalog brez dodatnih navodil. Lahko poskusite z bolj opisnimi pozivi kot tl;dr, da spremenite vedenje modela in prilagodite povzemanje, ki ga prejmete(3).  

Nedavne študije so pokazale znatne izboljšave na mnogih nalogah naravnega jezikovnega procesiranja in merilih z vnaprejšnjim učenjem na velikem korpusu besedila, čemur sledi fino prilagajanje na določeno nalogo. Čeprav je arhitektura običajno nalogovno nevtralna, ta metoda še vedno zahteva fino prilagajanje na specifične naloge z več tisoč ali deset tisoč primeri. Nasprotno pa ljudje na splošno lahko opravijo novo jezikovno nalogo že z nekaj primeri ali preprostimi navodili – nekaj, s čimer trenutni sistemi NLP še vedno večinoma težko shajajo. Tukaj pokažemo, da povečanje velikosti jezikovnih modelov močno izboljša nalogovno neodvisno delovanje z malo primeri, včasih celo doseže konkurenčnost z dosedanjimi najboljšimi pristopi fino prilagajanja.  



Povzeto  


# Vaje za več primerov uporabe  
1. Povzamemo besedilo  
2. Razvrstitev besedila  
3. Ustvarjanje novih imen izdelkov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Razvrsti besedilo  
#### Izziv  
Razvrsti predmete v kategorije, ki so podane ob izvajanju. V spodnjem primeru podamo tako kategorije kot besedilo za razvrstitev v pozivu (*playground_reference). 

Poizvedba stranke: Pozdravljeni, ena tipka na moji tipkovnici prenosnika se je pred kratkim zlomila in potreboval bom nadomestno:

Razvrščena kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Ustvari nove imena izdelkov
#### Izziv
Ustvarite imena izdelkov iz primerov besed. Tukaj v pozivu vključimo informacije o izdelku, za katerega bomo ustvarjali imena. Prav tako ponudimo podoben primer, da pokažemo vzorec, ki ga želimo dobiti. Temperaturo smo nastavili visoko, da povečamo naključnost in bolj inovativne odzive.

Opis izdelka: Domači aparat za pripravo mlečnih napitkov
Izvorne besede: hiter, zdrav, kompakten.
Imenska izdelkov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis izdelka: Par čevljev, ki ustrezajo katerikoli velikosti stopala.
Izvorne besede: prilagodljiv, prilegajoč, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Viri  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najboljše prakse za fino prilagajanje GPT modelov za razvrščanje besedila](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za več pomoči  
[Ekipa za komercializacijo OpenAI](AzureOpenAITeam@microsoft.com) 


# Sodelavci
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
